# 05_heart_merged_labels

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/Heart_Combination_3(Sketch harmony_RCTD_combined_cellnaming).ipynb`

Requires RCTD outputs, cluster-named Seurat object and four Valve CSVs.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
library(spacexr)

paths <- c(
  Heart_Young_1 = "RCTD_fullref_cutoff30_Heart_Young_1.rds",
  Heart_Young_2 = "RCTD_fullref_cutoff30_Heart_Young_2.rds",
  Heart_Aged_1  = "RCTD_fullref_cutoff30_Heart_Aged_1.rds",
  Heart_Aged_2  = "RCTD_fullref_cutoff30_Heart_Aged_2.rds"
)

# 以前あなたが出していた cutoff30 上限（Seurat側）
expected_cutoff30 <- c(
  Heart_Young_1 = 309139,
  Heart_Young_2 = 292681,
  Heart_Aged_1  = 327324,
  Heart_Aged_2  = 294701
)

summ_one <- function(nm, fp){
  r <- readRDS(fp)
  n_spots <- ncol(r@spatialRNA@counts)
  n_res   <- nrow(r@results$results_df)
  n_ref   <- ncol(r@reference@counts)
  nUMI    <- r@spatialRNA@nUMI
  tab     <- table(r@results$results_df$spot_class)
  tabn <- setNames(as.integer(tab[c("reject","singlet","doublet_certain","doublet_uncertain")]),
                   c("reject","singlet","doublet_certain","doublet_uncertain"))
  tabn[is.na(tabn)] <- 0L

  data.frame(
    sample = nm,
    file = fp,
    n_spots_in_counts = n_spots,
    n_rows_results_df = n_res,
    ok_counts_eq_results = (n_spots == n_res),
    ref_cells = n_ref,
    min_nUMI = min(nUMI),
    median_nUMI = unname(median(nUMI)),
    mean_nUMI = unname(mean(nUMI)),
    max_nUMI = max(nUMI),
    reject = tabn["reject"],
    singlet = tabn["singlet"],
    doublet_certain = tabn["doublet_certain"],
    doublet_uncertain = tabn["doublet_uncertain"],
    stringsAsFactors = FALSE
  )
}

res <- do.call(rbind, Map(summ_one, names(paths), paths))

# cutoff30上限との差（ほぼ0なら「30まで拾えている」）
res$expected_cutoff30 <- expected_cutoff30[res$sample]
res$delta_to_expected <- res$n_spots_in_counts - res$expected_cutoff30

res[order(res$sample), ]


Source cell 4 (one-based); original execution count 4.

In [ ]:
# 前提：paths と cellid_list はすでに作成済み
# paths <- c(Heart_Young_1=..., ...)

library(spacexr)

read_one_rctd <- function(sample, fp){
  r <- readRDS(fp)

  cellid <- colnames(r@spatialRNA@counts)

  # 予測タイプ：weights最大
  W <- r@results$weights
  pred_type <- colnames(W)[max.col(W, ties.method = "first")]

  # reject -> Unknown
  spot_class <- as.character(r@results$results_df$spot_class)
  pred_type2 <- pred_type
  pred_type2[spot_class == "reject"] <- "Unknown"

  data.frame(
    cellid = cellid,
    sample = sample,
    spot_class = spot_class,
    rctd_type = pred_type,
    rctd_type_unknown = pred_type2,
    stringsAsFactors = FALSE
  )
}

rctd_merged <- do.call(
  rbind,
  Map(read_one_rctd, names(paths), paths)
)

# 最低限の確認
stopifnot(!anyDuplicated(rctd_merged$cellid))
table(rctd_merged$sample)
table(rctd_merged$spot_class)

# Unknownがどれくらいか
table(rctd_merged$rctd_type_unknown)[c("Unknown")]

# 保存（次ステップでSeuratに付与）
saveRDS(rctd_merged, file = "RCTD_merged_cutoff30_rejectUnknown.rds")


Source cell 6 (one-based); original execution count 6.

In [ ]:
library(Seurat)

# 1) Seurat 読み込み
obj <- readRDS("heart_mergeobj_preProject_full_sketchHarmony_ENSMUSG16kept_clusterNaming.rds")

# 2) cellid一致チェック（rctd_merged は前ステップで作成済み前提）
seu_ids  <- colnames(obj)
rctd_ids <- rctd_merged$cellid

c(
  n_seurat = length(seu_ids),
  n_rctd   = length(rctd_ids),
  n_intersect = length(intersect(seu_ids, rctd_ids)),
  n_rctd_not_in_seurat = length(setdiff(rctd_ids, seu_ids)),
  n_seurat_not_in_rctd = length(setdiff(seu_ids, rctd_ids))
)

# 3) もしズレがあれば、例を数個だけ
head(setdiff(rctd_ids, seu_ids), 10)
head(setdiff(seu_ids, rctd_ids), 10)


Source cell 7 (one-based); original execution count 7.

In [ ]:
# 前提：obj と rctd_merged が既に存在

# 1) 付与用の named vector を作る（cellidを名前にする）
spotclass_vec <- setNames(rctd_merged$spot_class, rctd_merged$cellid)
type_vec      <- setNames(rctd_merged$rctd_type_unknown, rctd_merged$cellid)  # reject->Unknown

# 2) Seuratのcolnames順に並べて meta.data へ追加
obj$rctd_spot_class <- unname(spotclass_vec[colnames(obj)])
obj$rctd_type       <- unname(type_vec[colnames(obj)])

# 3) 付与できた数の確認（NAが 17645 になるはず）
c(
  n_seurat = ncol(obj),
  n_rctd_assigned_type = sum(!is.na(obj$rctd_type)),
  n_na_type = sum(is.na(obj$rctd_type)),
  n_rctd_assigned_class = sum(!is.na(obj$rctd_spot_class)),
  n_na_class = sum(is.na(obj$rctd_spot_class))
)

# 4) 値の内訳（NA除外）
table(obj$rctd_spot_class, useNA = "ifany")
table(obj$rctd_type, useNA = "ifany")[1:30]


Source cell 15 (one-based); original execution count 17.

In [ ]:
# 1) まず rctd_assigned を作成（既に作っていれば再実行OK）
obj$rctd_assigned <- !is.na(obj$rctd_type)

# 2) 解析対象：rctd_assigned==TRUE かつ rctd_type が NA ではない（念のため）
df_use <- subset(obj@meta.data, rctd_assigned & !is.na(rctd_type))

# 3) 新しい naming 列：cluster_naming を優先、NAのときだけ rctd_type を採用
df_use$new_naming <- as.character(df_use$cluster_naming)
idx_na <- is.na(df_use$new_naming) | df_use$new_naming == ""
df_use$new_naming[idx_na] <- as.character(df_use$rctd_type)[idx_na]

# 4) 確認（NAが残っていないか）
c(
  n_rows = nrow(df_use),
  n_new_naming_NA = sum(is.na(df_use$new_naming)),
  n_cluster_naming_NA = sum(is.na(df_use$cluster_naming)),
  n_rctd_type_NA = sum(is.na(df_use$rctd_type))
)

# 5) new_naming の内訳（多い順）
sort(table(df_use$new_naming), decreasing = TRUE)


Source cell 19 (one-based); original execution count 24.

In [ ]:
# df_use が存在している前提（new_naming列あり）

map_new <- c(
  B_cell_lineage = "B cell lineage",
  Cardiac_muscle_cell = "Ventricular cardiac muscle",
  Cardiac_neuron = "Cardiac neuron",
  Endocardial_cell = "Endothelial cell",
  Endothelial_cell = "Endothelial cell",
  Erythroid_cell = "Red blood cell",
  Fibroblast = "Fibroblast",
  Lymphatic_endothelial = "Lymphatic endothelial cell",
  Macrophage___Monocyte___DC = "Macrophage / Monocyte / DC",
  Mast_cell = "Mast cell",
  Pericyte = "VSMC / Pericyte",
  T_cell_lineage = "T cell lineage",
  Unknown = "Unknown",
  VSMC = "VSMC / Pericyte"
)

# 置換（該当するものだけ置き換える）
idx <- df_use$new_naming %in% names(map_new)
df_use$new_naming[idx] <- unname(map_new[df_use$new_naming[idx]])

# 確認
sort(unique(df_use$new_naming))


Source cell 20 (one-based); original execution count 25.

In [ ]:
# 1) Valve CSV を4つ読む（同じフォルダにある前提）
valve_files <- list.files(pattern = "^Heart_.*_Valve\\.csv$", full.names = TRUE)
valve_files

# 2) 1ファイルずつ：1列目(base id)を full cellid に変換して data.frame 化
read_valve_one <- function(f){
  sample <- sub("_Valve\\.csv$", "", basename(f))  # Heart_Aged_1_Valve.csv -> Heart_Aged_1
  x <- read.csv(f, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE)

  base_id <- x[[1]]                 # 1列目
  base_id <- gsub("^'+|'+$", "", base_id)  # もし'...'みたいに入ってたら除去（念のため）

  data.frame(
    sample = sample,
    base_id = base_id,
    cellid = paste0(sample, "_", base_id),
    stringsAsFactors = FALSE
  )
}

valve_list <- lapply(valve_files, read_valve_one)

# 3) まず各ファイルの行数だけ確認
sapply(valve_list, nrow)

# 4) 結合（この時点で“full cellid”は重複しないはず）
valve_df <- do.call(rbind, valve_list)

# 5) 重複チェック（full cellid）
sum(duplicated(valve_df$cellid))
head(valve_df, 5)


Source cell 21 (one-based); original execution count 26.

In [ ]:
# 前提：df_use（new_naming列あり）, valve_df（cellid列あり）が存在

# 1) df_useにcellid列（なければ作る）
if (!"cellid" %in% colnames(df_use)) df_use$cellid <- rownames(df_use)

# 2) Valve cellid集合
valve_ids <- unique(valve_df$cellid)

# 3) Valveに含まれる & Fibroblast のみ Cardiac valve に置換
idx <- df_use$cellid %in% valve_ids & df_use$new_naming == "Fibroblast"
sum(idx)  # 置換される数（ログ）
df_use$new_naming[idx] <- "Cardiac valve"

# 4) 確認（Valveセルの new_naming 内訳）
table(df_use$new_naming[df_use$cellid %in% valve_ids])


Source cell 24 (one-based); original execution count 29.

In [ ]:
# 1) 列名を Merged_cellname に（df_use前提）
df_use$Merged_cellname <- df_use$new_naming

# 2) 指定順に factor 化（この順で並ぶ）
lvl <- c(
  "Ventricular cardiac muscle",
  "Atrial cardiac muscle",
  "Endothelial cell",
  "Epicardium",
  "Endocardium",
  "Fibroblast",
  "Cardiac valve",
  "VSMC / Pericyte",
  "Lymphatic endothelial cell",
  "Cardiac neuron",
  "Airway epithelium",
  "Airway structural tissue",
  "Adipose tissue",
  "Macrophage / Monocyte / DC",
  "T cell lineage",
  "B cell lineage",
  "Mast cell",
  "Red blood cell",
  "Platelet",
  "Unknown"
)

df_use$Merged_cellname <- factor(df_use$Merged_cellname, levels = lvl)

# 3) 確認（並び順どおりに出る）
table(df_use$Merged_cellname)


Source cell 25 (one-based); original execution count 30.

In [ ]:
# 0) 念のため：cellid列
if (!"cellid" %in% colnames(obj@meta.data)) obj$cellid <- colnames(obj)

# 1) まず obj 側に空の列を作る（全部 NA で初期化）
obj$Merged_cellname <- NA

# 2) df_use は既に作成済み前提（rctd_assigned==TRUE & !is.na(rctd_type) で subset済み）
#    df_use の rownames は cellid のはず（違うなら df_use$cellid を使ってね）
stopifnot(all(rownames(df_use) %in% colnames(obj)))

# 3) df_use で確定した Merged_cellname を obj に書き戻す
obj$Merged_cellname[rownames(df_use)] <- as.character(df_use$Merged_cellname)

# 4) 確認（NA残りの内訳を軽く）
c(
  n_total = ncol(obj),
  n_assigned = sum(!is.na(obj$Merged_cellname)),
  n_na = sum(is.na(obj$Merged_cellname))
)


Source cell 26 (one-based); original execution count 31.

In [ ]:
# 解析対象（17645を除外）
obj_use <- subset(obj, subset = !is.na(Merged_cellname))

# 確認
c(
  n_total = ncol(obj),
  n_use   = ncol(obj_use),
  n_drop  = ncol(obj) - ncol(obj_use)
)


Source cell 28 (one-based); original execution count 33.

In [ ]:
# 解析対象（Merged_cellname が付いたものだけ）を保存
saveRDS(obj_use, file = "heart_mergeobj_preProject_full_sketchHarmony_ENSMUSG16kept_clusterNaming_RCTDmerged_MergedCellname_ASSIGNEDonly.rds")
